# Recurrent Neural Networks — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Reuse one transition over time so a hidden state can carry sequence memory
The cells show state updates, vanishing gradients, sequence classification, parameter sharing and teacher forcing on synthetic sequences.

In [ ]:
x=np.array([1.,0.,1.,1.]); h=0.; hs=[]
for xt in x: h=np.tanh(0.8*h+0.6*xt); hs.append(h)
plt.figure(figsize=(6,3)); plt.plot(hs,'-o'); plt.ylim(0,1); plt.title('RNN hidden state accumulates evidence')
assert hs[-1]>hs[0]

In [ ]:
ws=[0.2,0.5,0.9]; T=np.arange(1,21)
plt.figure(figsize=(6,3)); [plt.plot(T, w**T, label=f'w={w}') for w in ws]; plt.legend(); plt.title('Backprop through time multiplies Jacobians')
assert 0.2**20 < 0.9**20

In [ ]:
seq=np.array([0,1,1,0,1]); h=0; outs=[]
for xt in seq: h=0.7*h+xt; outs.append(h)
y=int(h>1.5)
plt.figure(figsize=(6,3)); plt.step(range(len(seq)),outs,where='mid'); plt.axhline(1.5,ls='--',c='r'); plt.title('Sequence classification from final state')
assert y==1

In [ ]:
T=6; params_shared=3; params_unrolled=3*T
plt.figure(figsize=(5,3)); plt.bar(['shared RNN','separate per step'],[params_shared,params_unrolled]); plt.title('Parameter sharing controls sequence length cost')
assert params_unrolled/params_shared==T

In [ ]:
seq=np.array([1,0,1,0,1]); h=0; preds=[]
for xt in seq[:-1]: h=0.5*h+xt; preds.append(int(h>0.5))
plt.figure(figsize=(6,2)); plt.plot(seq[1:],'-o',label='target next'); plt.plot(preds,'-s',label='pred'); plt.legend(); plt.title('One-step sequence prediction')
assert len(preds)==4 and preds[0]==1